In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 46.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 25.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
import random
from datasets import Dataset
import torch
from collections import defaultdict
import numpy as np
import warnings
import re
from tqdm import tqdm

warnings.filterwarnings(
    "ignore",
    message="Both `max_new_tokens`"
)

In [ ]:
# TODO: Set model size, task, representation, and D here
model_size="t5-small"
task="add" # either "add" or "subtract"
representation="words" # either "digit", "character", "fixed character", "underscore", "words", "10based", "10ebased"
D = 2 # max number of digits
n_samples = 1000
output_dir_name="small-add-words-D2" # folder to save model in, label it "[model size] - [operation] - [representation] - D[D value]", make sure it goes into the "models" folder

### **Generating training and development sets with balanced sampling**

In [ ]:
def generate_add_labels(dataset, representation):
    converter = REPRESENTATION_MAP.get(representation, lambda D, n: str(n))
    return [converter(None, num1 + num2) for num1, num2 in dataset]

def generate_subtract_labels(dataset, representation):
    converter = REPRESENTATION_MAP.get(representation, lambda D, n: str(n))
    return [converter(None, num1 - num2) for num1, num2 in dataset]

def format_add_inputs(dataset):
  return [(f"What is {num1} plus {num2}?") for num1, num2 in dataset]

def format_subtract_inputs(dataset):
  return [(f"What is {num1} minus {num2}?") for num1, num2 in dataset]

In [ ]:
def convert_to_character(num):
  '''
  Convert num given in digits to character representation (separated by whitespace)
  Ex: 832 -> 8 3 2
  '''

  str_num = str(num)
  return " ".join(str_num)

def convert_to_fixed_character(D, num):
  '''
  Convert num given in digits to fixed character representation (separated by whitespace)
  Ex: 832 -> 0 8 3 2 if the max is 4 digits
  '''

  str_num = str(num).zfill(D)   # pad with leading zeros
  return ' '.join(str_num)

def convert_to_underscore(num):
  '''
  Convert num given in digits to underscore representation
  Ex: 832 -> 8_3_2
  '''

  str_num = str(num)
  return "_".join(str_num)

def convert_to_words(num):
    '''
    Convert num given in digits to word representation
    Ex: 832 -> eight hundred thirty-two
    '''

    if num == 0:
        return "zero"

    ones = ["", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
            "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen",
            "sixteen", "seventeen", "eighteen", "nineteen"]

    tens = ["", "", "twenty", "thirty", "forty", "fifty",
            "sixty", "seventy", "eighty", "ninety"]

    thousands = ["", "thousand", "million", "billion", "trillion",
                 "quadrillion", "quintillion", "sextillion",
                 "septillion", "octillion", "nonillion", "decillion"]

    def chunk_to_words(n):
        result = ""

        if n >= 100:
            result += ones[n // 100] + " hundred"
            n %= 100
            if n:
                result += " "

        if n >= 20:
            result += tens[n // 10]
            n %= 10
            if n:
                result += "-" + ones[n]
        elif n > 0:
            result += ones[n]

        return result

    words = []
    chunk_index = 0

    while num > 0:
        chunk = num % 1000
        if chunk:
            chunk_words = chunk_to_words(chunk)
            if thousands[chunk_index]:
                chunk_words += " " + thousands[chunk_index]
            words.append(chunk_words)
        num //= 1000
        chunk_index += 1

    return " ".join(reversed(words))

def convert_to_10based(num):
  '''
  Convert num given in digits to 10-based representation
  Ex: 832 -> 8 100 3 10 2
  '''
  if num == 0:
        return "0"


  str_num = str(num)
  length = len(str_num)

  parts = []

  for i, digit in enumerate(str_num):
      if digit != '0':
          power = length - i - 1
          if power == 0:
              parts.append(digit)
          else:
              parts.append(f"{digit} {10**power}")

  return " ".join(parts)

def convert_to_10ebased(num):
  '''
  Convert num given in digits to 10-based representation
  Ex: 832 -> 8 10e2 3 10e1 2 10e0
  '''

  str_num = str(num)
  length = len(str_num)

  parts = []

  for i, digit in enumerate(str_num):
      power = length - i - 1
      parts.append(f"{digit} 10e{power}")

  return " ".join(parts)

REPRESENTATION_MAP = {
  "character": lambda D, n: convert_to_character(n),
  "fixed character": lambda D, n: convert_to_fixed_character(D, n),
  "underscore": lambda D, n: convert_to_underscore(n),
  "words": lambda D, n: convert_to_words(n),
  "10based": lambda D, n: convert_to_10based(n),
  "10ebased": lambda D, n: convert_to_10ebased(n),
}

def apply_representation(data, D, representation):
  converter = REPRESENTATION_MAP.get(representation)
  if converter is None:
      return data

  return [(converter(D, a), converter(D, b)) for a, b in data]

In [ ]:
D=30
worst_case = (10**D - 1) + (10**D - 1)
worst_case_str = convert_to_10based(worst_case)
tokens = tokenizer(worst_case_str)["input_ids"]
print(f"Worst case output tokens: {len(tokens)}")

Worst case output tokens: 167


In [ ]:
def generate_train_dataset(D, task, representation, n_samples=1000):

  # Generate n_samples of 2 number inputs
  data = []
  for _ in range(n_samples):
      d = random.randint(2, D) # sample a number from [2, D]

      # randomly sample two numbers from [10**(d-1), 10**d - 1]
      low = 10**(d - 1)
      high = 10**d - 1

      num1 = random.randint(low, high)
      num2 = random.randint(low, high)

      data.append((num1, num2))

  # Generate labels as strings
  labels = []
  if task == "add":
    labels = generate_add_labels(data, representation)
  elif task == "subtract":
    labels = generate_subtract_labels(data, representation)

  # Convert number representation to proper representation if necessary
  data = apply_representation(data, D, representation)

  # Format inputs in form "What is [num1] [operation] [num2]?"
  data_formatted = []

  if task == "add":
    data_formatted = format_add_inputs(data)
  elif task == "subtract":
    data_formatted = format_subtract_inputs(data)

  return data_formatted, labels


In [ ]:
train_addition_input, train_addition_labels = generate_train_dataset(D=D, task=task, representation=representation, n_samples=n_samples)
dev_addition_input, dev_addition_labels = generate_train_dataset(D=D, task=task, representation=representation, n_samples=n_samples)
train_addition_input[0:10], train_addition_labels[0:10]

(['What is ninety-nine plus eighteen?',
  'What is fifty-two plus eighty-seven?',
  'What is sixty-three plus ninety-one?',
  'What is nineteen plus eighty-three?',
  'What is eleven plus thirty?',
  'What is fifty-four plus seventy-seven?',
  'What is ninety-two plus thirty-nine?',
  'What is twenty-two plus sixty-five?',
  'What is eighty-seven plus fifty-seven?',
  'What is fifty-three plus eighty?'],
 ['one hundred seventeen',
  'one hundred thirty-nine',
  'one hundred fifty-four',
  'one hundred two',
  'forty-one',
  'one hundred thirty-one',
  'one hundred thirty-one',
  'eighty-seven',
  'one hundred forty-four',
  'one hundred thirty-three'])

In [ ]:
train_dataset = Dataset.from_dict({
    "input": train_addition_input,
    "target": train_addition_labels
})

dev_dataset = Dataset.from_dict({
    "input": dev_addition_input,
    "target": dev_addition_labels
})

In [ ]:
# Getting started with the T5-small model
# From HuggingFace: https://huggingface.co/google-t5/t5-small
# https://huggingface.co/google-t5/t5-base

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model_name = model_size
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model.generation_config.max_length = None
model.generation_config.max_new_tokens = 32

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

model.to(device)

# Tokenize inputs and labels and prepare for model
def preprocess(example):
    model_inputs = tokenizer(
        example["input"],
        padding=True,
        truncation=True,
        max_length=64 # Colab memory struggles with anything higher
    )

    labels = tokenizer(
        example["target"],
        padding=True,
        truncation=True,
        max_length=64
    )

    labels_ids = labels["input_ids"]
    labels_ids = [
        -100 if t == tokenizer.pad_token_id else t
        for t in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

train_data_tokenized = train_dataset.map(preprocess)
train_data_tokenized = train_data_tokenized.remove_columns(["input", "target"]) # remove raw columns

dev_data_tokenized = dev_dataset.map(preprocess)
dev_data_tokenized = dev_data_tokenized.remove_columns(["input", "target"]) # remove raw columns

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  if preds.ndim == 3:
      preds = np.argmax(preds, axis=-1)

  preds = np.where(
      (preds < 0) | (preds >= tokenizer.vocab_size),
      tokenizer.pad_token_id,
      preds
  )

  # replace -100 so labels can be decoded
  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

  decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  # EXACT MATCHING
  correct = sum(p.strip().lower() == l.strip().lower() for p, l in zip(decoded_preds, decoded_labels))
  return {"accuracy": correct / len(decoded_labels)}

In [ ]:
# Set up training
training_args = Seq2SeqTrainingArguments(
    output_dir="./" + output_dir_name, # TODO: change output_dir when training a new model
    per_device_train_batch_size=32, # paper uses batch size of 128 but no space on Colab to do this :(
    gradient_accumulation_steps=4,  # 32 * 4 = 128 effective batch size
    learning_rate=0.0003, # paper was 0.0003
    num_train_epochs=100,

    predict_with_generate=True,

    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=5 * (n_samples // 32),  # every 5 epochs worth of steps
    save_steps=5 * (n_samples // 32),

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data_tokenized,
    eval_dataset=dev_data_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

batch = next(iter(trainer.get_train_dataloader()))

print("INPUT IDS:\n", batch["input_ids"][0])
print("\nLABELS:\n", batch["labels"][0])

print("\nDECODED INPUT:\n", tokenizer.decode(batch["input_ids"][0], skip_special_tokens=True))
print("\nDECODED LABEL:\n", tokenizer.decode(
    [t if t != -100 else tokenizer.pad_token_id for t in batch["labels"][0]],
    skip_special_tokens=True
))

trainer.train()

INPUT IDS:
 tensor([  363,    19, 13369,   303,  2641,    63,    58,     1,     0,     0,
            0,     0,     0,     0,     0], device='cuda:0')

LABELS:
 tensor([4169,   17,   63,   18, 8264,    1, -100], device='cuda:0')

DECODED INPUT:
 What is twelve plus eighty?

DECODED LABEL:
 ninety-two


Step,Training Loss,Validation Loss,Accuracy
155,2.597730,0.591143,0.117000
310,1.865828,0.432429,0.273000
465,1.336838,0.312098,0.452000
620,1.071157,0.251501,0.557000
775,0.943124,0.233025,0.596000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=800, training_loss=1.90241037607193, metrics={'train_runtime': 328.4686, 'train_samples_per_second': 304.443, 'train_steps_per_second': 2.436, 'total_flos': 420236293570560.0, 'train_loss': 1.90241037607193, 'epoch': 100.0})

### **Save Model for Later**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
trainer.save_model("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name) # TODO: change path based on model ([model size] - [operation] - [representation])
tokenizer.save_pretrained("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/cs4782/Final/models/small-add-words-D2/tokenizer_config.json',
 '/content/drive/MyDrive/cs4782/Final/models/small-add-words-D2/tokenizer.json')

### **Computing Model Accuracy**

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")

    # move inputs to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # use greedy decoding
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=1, # greedy
        do_sample=False, # no randomness, we want greedy
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sixty-eight


In [ ]:
# Generate test sets with random sampling
def generate_test_dataset(D, task, representation, n_samples=1000):

  # Generate n_samples of 2 number inputs
  raw_data = []
  for _ in range(n_samples):
      # randomly sample two numbers from [0, 10**D - 1]
      high = 10**D - 1

      num1 = random.randint(0, high)
      num2 = random.randint(0, high)

      raw_data.append((num1, num2))

  # Generate labels as strings
  labels = []
  if task == "add":
    labels = generate_add_labels(raw_data, representation)
  elif task == "subtract":
    labels = generate_subtract_labels(raw_data, representation)

  # Convert number representation to proper representation if necessary
  data = apply_representation(raw_data, D, representation)

  # Format inputs in form "What is [num1] [operation] [num2]?"
  data_formatted = []
  if task == "add":
    data_formatted = format_add_inputs(data)
  elif task == "subtract":
    data_formatted = format_subtract_inputs(data)

  return raw_data, data_formatted, labels


In [ ]:
test_data, test_inputs, test_labels = generate_test_dataset(D=D, task=task, representation=representation, n_samples=1000)

In [ ]:
def evaluate_model_exact(test_inputs, test_labels):
    correct = 0
    total = len(test_inputs)

    for inp, label in tqdm(zip(test_inputs, test_labels), total=total):
        pred_text = predict(inp)

        # match compute_metrics behavior
        pred = pred_text.strip().lower()
        label = label.strip().lower()

        if pred == label:
            correct += 1

    return correct / total


acc = evaluate_model_exact(test_inputs, test_labels)
print()
print(f"Accuracy: {acc:.4f}")

100%|██████████| 1000/1000 [01:12<00:00, 13.79it/s]


Accuracy: 0.2990


### **Reload Previously-Saved Model**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Loading model back

model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name)
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()